In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / 'pyproject.toml').exists() and (path / 'src' / 'frb_isotropy').exists():
            return path
    raise FileNotFoundError('Could not find frb-isotropy project root from current directory')


PROJECT_ROOT = find_project_root()
SRC = PROJECT_ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from frb_isotropy.core.config import AnalysisConfig, create_runtime_context
from frb_isotropy.catalog.catalog import load_catalog, apply_mask
from frb_isotropy.selection.selection import build_survey_selection_functions
from frb_isotropy.simulations.injection import run_injection_recovery_suite
from frb_isotropy.visualization.plotting import plot_injection_recovery


project_root = PROJECT_ROOT
outputs_root = PROJECT_ROOT / "outputs" / "injection_recovery"
catalog_path = PROJECT_ROOT / "data" / "SkyPosition.csv"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("SRC =", SRC)
print("python =", sys.executable)
print("catalog_path =", catalog_path, "exists=", catalog_path.exists())

config = AnalysisConfig(
    name='SFAbsorptionInjectionRecovery',
    project_root=project_root,
    outputs_root=outputs_root,
    catalog_path=catalog_path,
    use_gal_mask=True,
    use_sel_func=True,
    n_jobs=100,
    gal_cut=20.0,
    min_sep=0.0,
    max_sep=180.0,
    bin_size=10,
    coarse_bins=(0.0, 20.0, 40.0, 60.0, 80.0, 100.0, 120.0, 140.0, 160.0, 180.0),
    nside_sf=32,
    smooth_sigma=3.0,
    perturbation_scale=1.0,
    n_rand_factor=20,
    n_ensemble=20,
    n_mocks_per_ensemble=50,
    random_seed=12345,
    mock_seed_base=100000,
    random_catalog_seed_base=200000,
    sf_perturbation_seed_base=300000,
    jackknife_seed_base=400000,
    bootstrap_seed_base=500000,
    selection_function_floor=1e-12,
    numerical_eigenvalue_floor=1e-15,
    stable_eigenvalue_floor=1e-12,
    numerical_zero_tolerance=1e-30,
    svd_eigenvalue_cut=1e-2,
    nside_jackknife=4,
    min_jackknife_regions=20,
    n_bootstrap=500,
    overlap_radius_deg=5.0,
    overlap_nside=32,
    run_top_survey_maps=True,
)

context = create_runtime_context(config)

df_data = load_catalog(context)
df_data = apply_mask(context, df_data)

selection_functions = build_survey_selection_functions(
    context=context,
    df_data=df_data,
    nside=config.nside_sf,
    smooth_sigma=config.smooth_sigma,
)

outputs = context.outputs


PROJECT_ROOT = /home/brunowesley/projetos/FRB-isotropy-tests/Codes/frb-isotropy
SRC = /home/brunowesley/projetos/FRB-isotropy-tests/Codes/frb-isotropy/src
python = /home/brunowesley/projetos/venv/bin/python
catalog_path = /home/brunowesley/projetos/FRB-isotropy-tests/Codes/frb-isotropy/data/SkyPosition.csv exists= True

--- Catalog loaded ---
Objects: 4066

--- Galactic mask applied ---
gal_cut = ±20.0 deg
Remaining objects: 2518
Removed objects: 1548

--- Survey partitioning ---
Unique surveys: 19

Survey populations:
          CHIME | N=2341 |  92.97%
         FRBCAT | N=  77 |   3.06%
          CRAFT | N=  33 |   1.31%
         PARKES | N=  22 |   0.87%
          ALERT | N=  15 |   0.60%
       MeerTRAP | N=   8 |   0.32%
           FAST | N=   5 |   0.20%
     Pan-STARRS | N=   4 |   0.16%
          ATLAS | N=   3 |   0.12%
   VLA-realfast | N=   2 |   0.08%
     GaiaAlerts | N=   2 |   0.08%
         UTMOST | N=   2 |   0.08%
            ZTF | N=   1 |   0.04%
        Tianlai | N=

In [2]:
# -----------------------------------------------------------------
# Empirical-SF absorption test
# -----------------------------------------------------------------

epsilons = np.linspace(0.0, 1.0, 11)

suite_dipolo = run_injection_recovery_suite(
    context=context,
    df_data=df_data,
    sf_set=selection_functions,
    epsilons=epsilons,
    multipole=1,
    n_realizations=20,
    axis_ra_deg=0.0,
    axis_dec_deg=90.0,
    verbose_sf=False,
)


Injection-recovery:   0%|          | 0/11 [00:00<?, ?epsilon/s]

  epsilon=0.00:   0%|          | 0/20 [00:00<?, ?it/s]

  epsilon=0.10:   0%|          | 0/20 [00:00<?, ?it/s]

  epsilon=0.20:   0%|          | 0/20 [00:00<?, ?it/s]

  epsilon=0.30:   0%|          | 0/20 [00:00<?, ?it/s]

...

  epsilon=0.70:   0%|          | 0/20 [00:00<?, ?it/s]

  epsilon=0.80:   0%|          | 0/20 [00:00<?, ?it/s]

  epsilon=0.90:   0%|          | 0/20 [00:00<?, ?it/s]

  epsilon=1.00:   0%|          | 0/20 [00:00<?, ?it/s]

epsilon=0.00 | signal no SF=4.441e-17 | fixed SF=6.776e-21 | rebuilt SF=8.403e-20 | retention rebuilt/fixed=nan% | absorbed=nan%
epsilon=0.10 | signal no SF=0.03455 | fixed SF=5.208e-05 | rebuilt SF=-2.885e-05 | retention rebuilt/fixed=-55.41% | absorbed=155.41%
epsilon=0.20 | signal no SF=0.06844 | fixed SF=0.0006196 | rebuilt SF=0.0001123 | retention rebuilt/fixed=18.12% | absorbed=81.88%
epsilon=0.30 | signal no SF=0.09863 | fixed SF=0.001606 | rebuilt SF=1.085e-05 | retention rebuilt/fixed=0.68% | absorbed=99.32%
epsilon=0.40 | signal no SF=0.1487 | fixed SF=0.003167 | rebuilt SF=0.0002314 | retention rebuilt/fixed=7.31% | absorbed=92.69%
epsilon=0.50 | signal no SF=0.1755 | fixed SF=0.005301 | rebuilt SF=0.0002257 | retention rebuilt/fixed=4.26% | absorbed=95.74%
epsilon=0.60 | signal no SF=0.2147 | fixed SF=0.008025 | rebuilt SF=0.0003177 | retention rebuilt/fixed=3.96% | absorbed=96.04%
epsilon=0.70 | signal no SF=0.2419 | fixed SF=0.01151 | rebuilt SF=0.000119 | retention rebui

In [3]:
# -----------------------------------------------------------------
# Numeric outputs
# -----------------------------------------------------------------

injection_summary = pd.DataFrame([
    {
        'epsilon_injected': r.epsilon_injected,
        'signal_no_sf': r.signal_no_sf,
        'signal_no_sf_std': r.signal_no_sf_std,
        'signal_fixed_sf': r.signal_fixed_sf,
        'signal_fixed_sf_std': r.signal_fixed_sf_std,
        'signal_rebuilt_sf': r.signal_rebuilt_sf,
        'signal_rebuilt_sf_std': r.signal_rebuilt_sf_std,
        'retention_fixed_vs_no_sf': r.retention_fixed_vs_no_sf,
        'retention_rebuilt_vs_fixed': r.retention_rebuilt_vs_fixed,
        'absorption_fraction': r.absorption_fraction,
        'raw_no_sf': r.raw_no_sf,
        'raw_fixed_sf': r.raw_fixed_sf,
        'raw_rebuilt_sf': r.raw_rebuilt_sf,
        'baseline_no_sf': r.baseline_no_sf,
        'baseline_fixed_sf': r.baseline_fixed_sf,
        'baseline_rebuilt_sf': r.baseline_rebuilt_sf,
    }
    for r in suite_dipolo.results
])

outputs.tables_dir.mkdir(parents=True, exist_ok=True)
summary_path = outputs.tables_dir / "injection_recovery_dipole_absorption_summary.csv"
injection_summary.to_csv(summary_path, index=False)

print("Saved summary table:", summary_path)
injection_summary


Saved summary table: /home/brunowesley/projetos/FRB-isotropy-tests/Codes/frb-isotropy/outputs/SFAbsorptionInjectionRecovery_mask_sf_gal20_bin10_nside32_smooth3/tables/injection_recovery_dipole_absorption_summary.csv


,epsilon_injected,signal_no_sf,signal_no_sf_std,signal_fixed_sf,signal_fixed_sf_std,signal_rebuilt_sf,signal_rebuilt_sf_std,retention_fixed_vs_no_sf,retention_rebuilt_vs_fixed,absorption_fraction,raw_no_sf,raw_fixed_sf,raw_rebuilt_sf,baseline_no_sf,baseline_fixed_sf,baseline_rebuilt_sf
0,0.0,4.440892e-17,0.039227,6.776264e-21,0.001256,8.402567e-20,0.000338,NaN,NaN,NaN,1.103476,0.000180,-0.000942,1.103476,0.00018,-0.000942
1,0.1,3.455206e-02,0.033478,5.207712e-05,0.001856,-2.885444e-05,0.000349,0.001507,-0.554071,1.554071,1.138028,0.000232,-0.000971,1.103476,0.00018,-0.000942
2,0.2,6.844409e-02,0.045647,6.196085e-04,0.002355,1.122641e-04,0.000443,0.009053,0.181186,0.818814,1.171920,0.000799,-0.000829,1.103476,0.00018,-0.000942
3,0.3,9.862772e-02,0.047599,1.606081e-03,0.003283,1.084570e-05,0.000243,0.016284,0.006753,0.993247,1.202103,0.001786,-0.000931,1.103476,0.00018,-0.000942
4,0.4,1.486921e-01,0.032091,3.166921e-03,0.003220,2.314384e-04,0.000380,0.021299,0.073080,0.926920,1.252168,0.003347,-0.000710,1.103476,0.00018,-0.000942
5,0.5,1.754624e-01,0.036002,5.301458e-03,0.003252,2.256589e-04,0.000370,0.030214,0.042565,0.957435,1.278938,0.005481,-0.000716,1.103476,0.00018,-0.000942
6,0.6,2.146542e-01,0.045102,8.024580e-03,0.004104,3.176704e-04,0.000413,0.037384,0.039587,0.960413,1.318130,0.008204,-0.000624,1.103476,0.00018,-0.000942
7,0.7,2.418545e-01,0.043372,1.151355e-02,0.005376,1.189668e-04,0.000375,0.047605,0.010333,0.989667,1.345330,0.011693,-0.000823,1.103476,0.00018,-0.000942
8,0.8,2.601507e-01,0.035423,1.462802e-02,0.003844,1.739788e-04,0.000430,0.056229,0.011894,0.988106,1.363626,0.014808,-0.000768,1.103476,0.00018,-0.000942
9,0.9,2.951537e-01,0.054439,2.076135e-02,0.005281,1.089981e-04,0.000444,0.070341,0.005250,0.994750,1.398629,0.020941,-0.000833,1.103476,0.00018,-0.000942


In [4]:
outputs.figures_dir.mkdir(parents=True, exist_ok=True)
figure_path = outputs.figures_dir / "injection_recovery_dipole_absorption.png"

plot_injection_recovery(
    suite_dipolo,
    output_path=figure_path,
)

print("Saved figure:", figure_path)


Saved figure: /home/brunowesley/projetos/FRB-isotropy-tests/Codes/frb-isotropy/outputs/SFAbsorptionInjectionRecovery_mask_sf_gal20_bin10_nside32_smooth3/figures/injection_recovery_dipole_absorption.png
